# Notebook 4 — Sector Clustering
**Purpose:** Use unsupervised ML to find clusters of companies with similar financial profiles.
Methods: K-Means (K=4,5,6 with Elbow), DBSCAN, PCA visualization.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, sqlite3, os, sys
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
sys.path.insert(0, os.path.abspath('..'))
sns.set_theme(style='darkgrid'); plt.rcParams['figure.figsize'] = (14, 6)

conn = sqlite3.connect(os.path.join('..', 'db.sqlite3'))
companies = pd.read_sql('SELECT c.*, s.sector_name FROM dim_company c LEFT JOIN dim_sector s ON c.sector_id=s.sector_id', conn)
years = pd.read_sql('SELECT * FROM dim_year ORDER BY sort_order', conn)
pl = pd.read_sql('SELECT * FROM fact_profit_loss', conn).merge(years[['year_id','sort_order']], on='year_id')
bs = pd.read_sql('SELECT * FROM fact_balance_sheet', conn).merge(years[['year_id','sort_order']], on='year_id')
cf = pd.read_sql('SELECT * FROM fact_cash_flow', conn).merge(years[['year_id','sort_order']], on='year_id')
analysis = pd.read_sql('SELECT * FROM fact_analysis', conn)
ml_scores = pd.read_sql('SELECT * FROM fact_ml_scores', conn)
print(f'Companies: {len(companies)}')

## Step 1: Feature Engineering

In [ ]:
latest_pl = pl.sort_values('sort_order').groupby('company_id').last().reset_index()
latest_bs = bs.sort_values('sort_order').groupby('company_id').last().reset_index()
latest_cf = cf.sort_values('sort_order').groupby('company_id').last().reset_index()
growth_3y = analysis[analysis['period_label']=='3Y'][['company_id','compounded_sales_growth_pct']]

features = companies[['symbol','company_name','sector_name','roe_pct']].copy()
features = features.merge(latest_pl[['company_id','opm_pct','dividend_payout_pct','net_profit','sales']], left_on='symbol', right_on='company_id', how='left')
features = features.merge(latest_bs[['company_id','debt_to_equity']], left_on='symbol', right_on='company_id', how='left', suffixes=('','_bs'))
features = features.merge(latest_cf[['company_id','operating_activity']], left_on='symbol', right_on='company_id', how='left', suffixes=('','_cf'))
features = features.merge(growth_3y.rename(columns={'compounded_sales_growth_pct':'growth_3y'}), left_on='symbol', right_on='company_id', how='left', suffixes=('','_an'))

features['cash_conversion'] = np.where(features['net_profit']>0, features['operating_activity']/features['net_profit'], 0)

FEATURE_COLS = ['opm_pct','roe_pct','growth_3y','debt_to_equity','cash_conversion','dividend_payout_pct']
feat_clean = features.dropna(subset=FEATURE_COLS).copy()
print(f'Companies with complete features: {len(feat_clean)}')

scaler = StandardScaler()
X = scaler.fit_transform(feat_clean[FEATURE_COLS])
print(f'Feature matrix shape: {X.shape}')

## Step 2: K-Means — Elbow Method

In [ ]:
inertias, sil_scores = [], []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-'); axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[1].plot(K_range, sil_scores, 'ro-'); axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Score')
best_k = list(K_range)[np.argmax(sil_scores)]
print(f'Best K by silhouette: {best_k}')
plt.tight_layout(); plt.show()

## Step 3: K-Means Clustering (K=5)

In [ ]:
km_final = KMeans(n_clusters=5, random_state=42, n_init=10)
feat_clean['kmeans_cluster'] = km_final.fit_predict(X)

# PCA for 2D visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
feat_clean['pca1'] = X_pca[:, 0]
feat_clean['pca2'] = X_pca[:, 1]

plt.figure(figsize=(12, 8))
for cluster in sorted(feat_clean['kmeans_cluster'].unique()):
    mask = feat_clean['kmeans_cluster'] == cluster
    plt.scatter(feat_clean.loc[mask,'pca1'], feat_clean.loc[mask,'pca2'], label=f'Cluster {cluster}', s=60, alpha=0.7)
    # Label some points
    for _, row in feat_clean[mask].head(3).iterrows():
        plt.annotate(row['symbol'], (row['pca1'], row['pca2']), fontsize=7, alpha=0.8)
plt.title('K-Means Clusters (PCA 2D)', fontweight='bold', fontsize=14)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.legend(); plt.tight_layout(); plt.show()

## Step 4: DBSCAN Clustering

In [ ]:
db = DBSCAN(eps=1.5, min_samples=3)
feat_clean['dbscan_cluster'] = db.fit_predict(X)
n_clusters_db = len(set(feat_clean['dbscan_cluster'])) - (1 if -1 in feat_clean['dbscan_cluster'].values else 0)
n_noise = (feat_clean['dbscan_cluster'] == -1).sum()
print(f'DBSCAN: {n_clusters_db} clusters, {n_noise} noise points')

plt.figure(figsize=(12, 8))
for cluster in sorted(feat_clean['dbscan_cluster'].unique()):
    mask = feat_clean['dbscan_cluster'] == cluster
    label = f'Cluster {cluster}' if cluster != -1 else 'Noise'
    plt.scatter(feat_clean.loc[mask,'pca1'], feat_clean.loc[mask,'pca2'], label=label, s=60, alpha=0.7)
plt.title('DBSCAN Clusters (PCA 2D)', fontweight='bold', fontsize=14)
plt.legend(); plt.tight_layout(); plt.show()

## Step 5: Label Clusters

In [ ]:
cluster_profiles = feat_clean.groupby('kmeans_cluster')[FEATURE_COLS].mean().round(2)
print('=== K-Means Cluster Profiles ===')
print(cluster_profiles.to_string())

# Auto-label based on dominant features
cluster_labels = {}
for c in cluster_profiles.index:
    p = cluster_profiles.loc[c]
    if p['growth_3y'] > cluster_profiles['growth_3y'].median() and p['debt_to_equity'] < cluster_profiles['debt_to_equity'].median():
        cluster_labels[c] = 'High Growth, Low Debt'
    elif p['dividend_payout_pct'] > cluster_profiles['dividend_payout_pct'].median() and p['opm_pct'] > cluster_profiles['opm_pct'].median():
        cluster_labels[c] = 'Stable Dividend, High Margin'
    elif p['debt_to_equity'] > cluster_profiles['debt_to_equity'].median():
        cluster_labels[c] = 'High Leverage'
    elif p['opm_pct'] < cluster_profiles['opm_pct'].median():
        cluster_labels[c] = 'Low Margin'
    else:
        cluster_labels[c] = 'Moderate Growth'

feat_clean['cluster_label'] = feat_clean['kmeans_cluster'].map(cluster_labels)
print('\nCluster Labels:', cluster_labels)

## Step 6: Compare Clusters to Sectors

In [ ]:
cross_tab = pd.crosstab(feat_clean['sector_name'], feat_clean['cluster_label'])
plt.figure(figsize=(14, 8))
sns.heatmap(cross_tab, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Sector vs Financial Cluster — Cross-Tabulation', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

print('\n=== Sector-Cluster Mapping ===')
for sector in cross_tab.index:
    dominant = cross_tab.loc[sector].idxmax()
    print(f'  {sector}: mostly {dominant}')

feat_clean[['symbol','company_name','sector_name','kmeans_cluster','cluster_label']].to_csv('../data/company_clusters.csv', index=False)
print(f'\n✅ Exported clustering results to data/company_clusters.csv')
conn.close()